In [1]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
#HDFC OHLC data 
stock = yf.download("HDFCBANK.NS", start="2025-04-01", end="2026-03-01")

stock.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,HDFCBANK.NS,HDFCBANK.NS,HDFCBANK.NS,HDFCBANK.NS,HDFCBANK.NS
Date,,,,,
2025-04-01,857.946411,878.353510,856.733149,874.519588,28511006
2025-04-02,872.044495,873.160660,859.329469,859.329469,11573564
2025-04-03,871.049622,875.490174,862.386940,864.376678,11250038
2025-04-04,881.944824,893.907545,878.935898,879.857989,33540454
2025-04-07,853.020508,863.770025,843.557061,855.495611,37995450


In [4]:
from sqlalchemy import create_engine


engine = create_engine("sqlite:///../database/bluestock_mf.db")

nav = pd.read_sql("select * from fact_nav",engine)
nav.head()

,Scheme_Code,Date,NAV
0,147711,2026-06-21,1001.6699
1,118698,2026-06-21,1004.3364
2,147453,2026-06-21,1004.1296
3,147452,2026-06-21,1216.5909
4,147451,2026-06-21,1003.0695


In [5]:
nav["Date"] = pd.to_datetime(nav["Date"])

In [2]:
import pandas as pd

nav = pd.read_csv("../data/raw/sample_historical_nav.csv")

nav["Date"] = pd.to_datetime(nav["Date"])

scheme_code = 120503

scheme = nav[nav["Scheme_Code"] == scheme_code]

scheme = scheme.sort_values("Date")

start_nav = scheme.iloc[0]["NAV"]

end_nav = scheme.iloc[-1]["NAV"]

years = (scheme["Date"].max() - scheme["Date"].min()).days / 365

cagr = ((end_nav / start_nav) ** (1 / years)) - 1

print("Scheme Code:", scheme_code)
print("Starting NAV:", start_nav)
print("Ending NAV:", end_nav)
print("Investment Period:", round(years,2),"years")
print("CAGR:", round(cagr * 100,2), "%")

Scheme Code: 120503
Starting NAV: 25.35
Ending NAV: 50.12
Investment Period: 5.01 years
CAGR: 14.59 %


In [ ]:
#CAGR calculations
cagr_results = []

for code in nav["Scheme_Code"].unique():
    scheme = nav[nav["Scheme_Code"] == code].sort_values("Date")

    if len(scheme) > 1:
        start_nav = scheme.iloc[0]["NAV"]
        end_nav = scheme.iloc[-1]["NAV"]
        years = (scheme["Date"].max() - scheme["Date"].min()).days / 365

        if years > 0:
            cagr = ((end_nav/start_nav)**(1/years)) - 1
            cagr_results.append([code,round(cagr*100,2)])

cagr_df = pd.DataFrame(cagr_results,columns=["Scheme_Code","CAGR_Percentage"])

cagr_df.head()

,Scheme_Code,CAGR_Percentage
0,120503,14.59
1,120504,13.81
2,120505,14.03
3,120506,11.96


In [5]:
#Linear Regression
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("sqlite:///../database/bluestock_mf.db")
performance = pd.read_sql("select * from fact_performance", engine)

performance.head()

,Scheme_Code,return_1y,return_3y,return_5y,sharpe,sortino,beta,alpha,max_drawdown,var95
0,100033,7.57,17.83,15.39,1.49,1.62,0.95,-0.69,-19.04,-2.70
1,100034,15.58,16.30,15.28,0.92,1.43,1.13,-0.47,-21.41,-4.93
2,100037,15.78,17.90,15.57,1.39,1.81,0.82,-0.14,-17.79,-4.81
3,100038,13.20,19.88,12.58,1.47,1.28,0.86,3.75,-22.76,-4.00
4,100039,15.04,18.31,15.53,1.18,1.45,0.88,-0.87,-20.72,-3.83


In [6]:
X = performance[["return_1y","return_3y","sharpe","beta"]]

y = performance["return_5y"]

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [14]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train,y_train)

prediction = model.predict(X_test)
prediction[:5]

array([15.19750587, 15.2215548 , 15.19750587, 15.55738046, 14.3657109 ])

In [ ]:
#accuracy
from sklearn.metrics import mean_absolute_error

error = mean_absolute_error(y_test,prediction)

print("Mean Absolute Error:",error)

Mean Absolute Error: 1.040346071379428
